# Raw-Text Embedding with Original `nomic-ai/nomic-embed-text-v1.5`

Embeds training-split essays using the **base** (non-finetuned) nomic model and saves a
FAISS index to `data/vector_db/essays_nomic_base/`.  
Use this notebook to establish a baseline; switch to the finetuned model later by changing `MODEL_NAME_OR_PATH`.

**Output files**
- `data/vector_db/essays_nomic_base/vectors.faiss`
- `data/vector_db/essays_nomic_base/vectors_meta.jsonl`

## CONFIG

In [ ]:
import os, sys

# ── adjust to your repo root if needed ──────────────────────────────────────
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# ── model ────────────────────────────────────────────────────────────────────
# Use the original base model here.  Swap for the local finetuned path later:
#   MODEL_NAME_OR_PATH = os.path.join(PROJECT_ROOT, 'models', 'sbert_essays_finetuned')
MODEL_NAME_OR_PATH = "nomic-ai/nomic-embed-text-v1.5"

# Sequence length — nomic architecture supports up to 8192;
# 2048 covers >99.7 % of essays with no truncation.
MAX_SEQ_LENGTH = 2048
SLIDING_STRIDE = 1024   # stride for the sliding-window when text > MAX_SEQ_LENGTH

# ── data paths ───────────────────────────────────────────────────────────────
TRAIN_CSV    = os.path.join(PROJECT_ROOT, "data", "split", "essays", "train.csv")
VAL_CSV      = os.path.join(PROJECT_ROOT, "data", "split", "essays", "val.csv")
OUTPUT_DIR   = os.path.join(PROJECT_ROOT, "data", "vector_db", "essays_nomic_base")

FORCE_REBUILD = False   # set True to overwrite an existing index

print(f"Project root : {PROJECT_ROOT}")
print(f"Model        : {MODEL_NAME_OR_PATH}")
print(f"Max seq len  : {MAX_SEQ_LENGTH}")
print(f"Output dir   : {OUTPUT_DIR}")

## Imports

In [ ]:
import time
import json
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

print("Imports OK")

## Load model and data

In [ ]:
model = SentenceTransformer(MODEL_NAME_OR_PATH, trust_remote_code=True)
model.max_seq_length = MAX_SEQ_LENGTH
print(f"Model loaded  : {MODEL_NAME_OR_PATH}")
print(f"max_seq_length: {model.max_seq_length}")

In [ ]:
train_df = pd.read_csv(TRAIN_CSV)
val_df   = pd.read_csv(VAL_CSV)
print(f"Train: {len(train_df)} rows    Val: {len(val_df)} rows")
train_df.head(2)

## Token-length audit
Confirm how many training essays exceed `MAX_SEQ_LENGTH` tokens.

In [ ]:
tokenizer = model.tokenizer

def count_tokens(text):
    return len(tokenizer.encode(str(text), add_special_tokens=True))

train_df["n_tokens"] = train_df["text"].apply(count_tokens)

print(f"Max tokens : {train_df['n_tokens'].max()}")
print(f"Mean tokens: {train_df['n_tokens'].mean():.0f}")
print(f"P95 tokens : {train_df['n_tokens'].quantile(0.95):.0f}")
print(f"P99 tokens : {train_df['n_tokens'].quantile(0.99):.0f}")
over = (train_df['n_tokens'] > MAX_SEQ_LENGTH).sum()
print(f"Essays > {MAX_SEQ_LENGTH} tokens: {over} ({over/len(train_df)*100:.2f}%)")

## Embedding helpers

Sliding-window mean pooling handles the rare essays that exceed `MAX_SEQ_LENGTH`.  
For essays within the limit, a single forward pass is used.

In [ ]:
def _embed_single(text: str) -> np.ndarray:
    """Embed one text string; use sliding window if tokens > MAX_SEQ_LENGTH."""
    tokens = tokenizer.encode(str(text), add_special_tokens=True)
    if len(tokens) <= MAX_SEQ_LENGTH:
        vec = model.encode(text, convert_to_numpy=True, normalize_embeddings=True)
        return vec.astype("float32")

    # Sliding-window: chunk with stride, embed each, mean-pool, re-normalise
    all_text = tokenizer.decode(tokens, skip_special_tokens=True)
    chunk_toks = []
    start = 0
    while start < len(tokens):
        chunk_toks.append(tokens[start : start + MAX_SEQ_LENGTH])
        start += SLIDING_STRIDE
        if start + MAX_SEQ_LENGTH >= len(tokens):
            # last chunk: grab the tail and stop
            if start < len(tokens):
                chunk_toks.append(tokens[start:])
            break

    chunk_texts = [tokenizer.decode(c, skip_special_tokens=True) for c in chunk_toks]
    vecs = model.encode(chunk_texts, convert_to_numpy=True, normalize_embeddings=True).astype("float32")
    pooled = vecs.mean(axis=0)
    norm = np.linalg.norm(pooled)
    return (pooled / norm if norm > 0 else pooled)


print("Embedding helpers defined.")

## Label extraction helper

In [ ]:
TRAIT_MAP = {
    "cEXT": "Extraversion",
    "cNEU": "Neuroticism",
    "cAGR": "Agreeableness",
    "cCON": "Conscientiousness",
    "cOPN": "Openness to Experience",
}

def extract_trait_labels(row):
    labels = {}
    for col, trait_name in TRAIT_MAP.items():
        if col in row.index:
            raw = str(row[col]).strip().lower()
            if raw in ("high", "low"):
                labels[trait_name] = raw
            elif raw in ("1", "1.0"):
                labels[trait_name] = "high"
            elif raw in ("0", "0.0"):
                labels[trait_name] = "low"
    return labels

print("Label helper defined.")

## Build FAISS index
Embeds all training essays and saves the index to `OUTPUT_DIR`.

In [ ]:
import faiss

index_path = os.path.join(OUTPUT_DIR, "vectors.faiss")
meta_path  = os.path.join(OUTPUT_DIR, "vectors_meta.jsonl")

if not FORCE_REBUILD and os.path.exists(index_path) and os.path.exists(meta_path):
    print(f"Index already exists at {OUTPUT_DIR!r}. Set FORCE_REBUILD=True to overwrite.")
else:
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    n   = len(train_df)
    t0  = time.time()
    embeddings = []
    meta       = []

    for i, row in train_df.iterrows():
        raw_text = str(row["text"])
        vec      = _embed_single(raw_text)
        embeddings.append(vec)
        meta.append({
            "user_id":      f"user_{i}",
            "posts_raw":    raw_text,
            "trait_labels": extract_trait_labels(row),
            "profile":      {},
        })
        if (i + 1) % 200 == 0:
            elapsed = time.time() - t0
            rate    = (i + 1) / elapsed
            eta     = (n - i - 1) / rate
            print(f"  {i+1}/{n}  elapsed={elapsed:.0f}s  ETA={eta:.0f}s")

    elapsed = time.time() - t0
    print(f"Embedded {n} essays in {elapsed:.1f}s")

    emb_arr = np.array(embeddings, dtype="float32")
    dim     = emb_arr.shape[1]

    # Inner-product index (vectors are already L2-normalised → IP == cosine sim)
    faiss_idx = faiss.IndexFlatIP(dim)
    faiss_idx.add(emb_arr)
    faiss.write_index(faiss_idx, index_path)

    with open(meta_path, "w", encoding="utf-8") as fout:
        for entry in meta:
            fout.write(json.dumps(entry, ensure_ascii=False) + "\n")

    print(f"Saved {n} vectors (dim={dim}) -> {OUTPUT_DIR}/")
    print(f"  vectors.faiss     : {os.path.getsize(index_path)/1e6:.1f} MB")
    print(f"  vectors_meta.jsonl: {os.path.getsize(meta_path)/1e6:.1f} MB")

## Sanity check: retrieval label accuracy on validation set

For each validation essay, retrieve the top-5 nearest training essays and check whether
the majority label of the retrieved set matches the true label.  
Run this after building the index to get a quick quality signal.

In [ ]:
import faiss as _faiss

# Load index + meta
_idx  = _faiss.read_index(index_path)
_meta = [json.loads(l) for l in open(meta_path, encoding="utf-8")]
print(f"Index loaded: {_idx.ntotal} vectors")

TOP_K      = 5
FETCH_N    = TOP_K * 6   # over-fetch to filter per trait
TRAIT_COLS = list(TRAIT_MAP.items())  # [(col, full_name), ...]

results = {trait: {"correct": 0, "total": 0} for _, trait in TRAIT_COLS}

for i, row in val_df.iterrows():
    q_vec = _embed_single(str(row["text"])).reshape(1, -1)
    _, idxs = _idx.search(q_vec, FETCH_N)

    for col, trait in TRAIT_COLS:
        if col not in row.index:
            continue
        true_raw = str(row[col]).strip().lower()
        if true_raw in ("1", "1.0"):
            true_label = "high"
        elif true_raw in ("0", "0.0"):
            true_label = "low"
        elif true_raw in ("high", "low"):
            true_label = true_raw
        else:
            continue

        retrieved_labels = []
        for idx_i in idxs[0]:
            if idx_i < 0:
                continue
            tl = _meta[idx_i]["trait_labels"].get(trait)
            if tl:
                retrieved_labels.append(tl)
            if len(retrieved_labels) >= TOP_K:
                break

        if not retrieved_labels:
            continue
        majority = "high" if retrieved_labels.count("high") > retrieved_labels.count("low") else "low"
        results[trait]["total"]   += 1
        results[trait]["correct"] += int(majority == true_label)

print(f"\nRetrieval label accuracy (top-{TOP_K} majority vote) — original nomic-ai base model")
print("-" * 55)
total_c, total_t = 0, 0
for trait, r in results.items():
    acc = r["correct"] / r["total"] * 100 if r["total"] else 0
    print(f"  {trait:<30s}  {acc:5.1f}%  ({r['correct']}/{r['total']})")
    total_c += r["correct"]
    total_t += r["total"]
print("-" * 55)
print(f"  {'Overall':<30s}  {total_c/total_t*100:5.1f}%  ({total_c}/{total_t})")

## Comparison note

| Index | Model | Embeddings used | Expected strength |
|---|---|---|---|
| `essays_nomic_base` (this notebook) | original `nomic-ai/nomic-embed-text-v1.5` | raw text only | General semantic similarity baseline |
| `essays_nomic_finetuned` | finetuned `sbert_essays_finetuned` | raw text only | Personality-domain adapted |
| `essays_dual` (build_dual_index.ipynb) | finetuned | raw + profile (dual fusion) | Richest representation |

Run the retrieval sanity check on all three indexes and compare per-trait accuracy to select the best backend.

## Switching to the finetuned model (later)

When you're ready to compare against the finetuned model, simply change the CONFIG cell:

```python
MODEL_NAME_OR_PATH = os.path.join(PROJECT_ROOT, 'models', 'sbert_essays_finetuned')
OUTPUT_DIR = os.path.join(PROJECT_ROOT, 'data', 'vector_db', 'essays_nomic_finetuned')
FORCE_REBUILD = True
```

Then re-run all cells.  The finetuned model already has `max_seq_length=2048` set in its
`sentence_bert_config.json`, so no further changes are needed.